# Table VI ED50 ranking and censoring check

This notebook separates **exact ED50 values** from **right-censored ED50 values** in the curated Table VI dataset.

The goal is not to claim that ED50 ranking alone explains the historical selection of E-1020 / 11a. The goal is to make the first pharmacology ranking transparent and teachable:

- exact ED50 values can be ranked directly;
- right-censored values such as `>300` or `>1000` should **not** be treated as exact values;
- `iv_potency_class` is only a coarse label for Table VI IV inotropic potency, not a claim of clinical value, safety, drug-likeness, or therapeutic superiority.


## 1. Setup

Run this notebook from either the repository root or the `notebooks/` directory. The path helper below tries both locations.


In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

candidate_paths = [
    Path("data/curated/e1020_table_vi_v0.csv"),
    Path("../data/curated/e1020_table_vi_v0.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find data/curated/e1020_table_vi_v0.csv. "
        "Run this notebook from the repository root or notebooks/ directory."
    )

print(f"Using dataset: {data_path}")

## 2. Load the curated Table VI dataset

This notebook expects the following ED50-related columns to exist:

- `ed50_ug_per_kg`
- `ed50_relation`
- `ed50_censored`

If `curation_status` exists, the notebook also summarizes it. This is useful after Phase 1 verification.


In [ ]:
df = pd.read_csv(data_path)

required_columns = [
    "compound_id",
    "ed50_ug_per_kg",
    "ed50_relation",
    "ed50_censored",
    "iv_potency_class",
]

missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Rows: {len(df)}")
print("Columns:")
print(list(df.columns))

df.head()

## 3. Quick curation and ED50 checks

Before ranking, check whether the dataset has been manually verified and whether ED50 censoring fields are internally consistent.


In [ ]:
if "curation_status" in df.columns:
    print("curation_status counts:")
    display(df["curation_status"].value_counts(dropna=False).rename_axis("curation_status").reset_index(name="count"))
else:
    print("No curation_status column found. This notebook can still run, but Phase 1 verification is recommended.")

print("\ned50_relation counts:")
display(df["ed50_relation"].value_counts(dropna=False).rename_axis("ed50_relation").reset_index(name="count"))

print("\ned50_censored counts:")
display(df["ed50_censored"].value_counts(dropna=False).rename_axis("ed50_censored").reset_index(name="count"))

In [ ]:
# Basic consistency checks between relation and censoring labels.
# Expected mapping:
#   ed50_relation == "=" -> ed50_censored == "none"
#   ed50_relation == ">" -> ed50_censored == "right"

expected_censoring = {"=": "none", ">": "right"}

check_df = df[["compound_id", "ed50_relation", "ed50_censored"]].copy()
check_df["expected_ed50_censored"] = check_df["ed50_relation"].map(expected_censoring)
check_df["censoring_consistent"] = (
    check_df["ed50_censored"] == check_df["expected_ed50_censored"]
)

inconsistent = check_df[~check_df["censoring_consistent"]]

if inconsistent.empty:
    print("ED50 relation and censoring labels are consistent.")
else:
    print("Potential inconsistencies found:")
    display(inconsistent)

## 4. Add a notebook-only `ed50_type` helper column

This derived column is created only inside the notebook. It avoids adding redundant columns to the curated CSV.


In [ ]:
def classify_ed50_type(row):
    if row["ed50_relation"] == "=" and row["ed50_censored"] == "none":
        return "exact"
    if row["ed50_relation"] == ">" and row["ed50_censored"] == "right":
        return "right_censored"
    return "unknown"

analysis_df = df.copy()
analysis_df["ed50_type"] = analysis_df.apply(classify_ed50_type, axis=1)

analysis_df["ed50_display"] = analysis_df.apply(
    lambda row: f'{row["ed50_relation"]}{row["ed50_ug_per_kg"]:g}',
    axis=1,
)

analysis_df[["compound_id", "ed50_display", "ed50_type", "iv_potency_class"]].head()

## 5. Separate exact and right-censored ED50 values

The core rule:

- `exact_ed50`: ED50 can be directly ranked by numeric value.
- `right_censored_ed50`: ED50 is only known to be greater than the reported threshold. For example, `>300` means **greater than 300**, not exactly 300.


In [ ]:
exact_ed50 = analysis_df[analysis_df["ed50_type"] == "exact"].copy()
right_censored_ed50 = analysis_df[analysis_df["ed50_type"] == "right_censored"].copy()
unknown_ed50 = analysis_df[analysis_df["ed50_type"] == "unknown"].copy()

print(f"Exact ED50 rows: {len(exact_ed50)}")
print(f"Right-censored ED50 rows: {len(right_censored_ed50)}")
print(f"Unknown ED50 rows: {len(unknown_ed50)}")

if not unknown_ed50.empty:
    print("\nRows requiring ED50 type review:")
    display(unknown_ed50[["compound_id", "ed50_ug_per_kg", "ed50_relation", "ed50_censored"]])

## 6. Exact ED50 ranking

Only exact ED50 values are ranked here. Lower ED50 indicates stronger IV inotropic potency in the Table VI context, but this is **not** an overall drug ranking.


In [ ]:
key_compound_roles = {
    "11a": "E-1020 candidate",
    "23": "7-yl isomer / weak comparator",
    "Milrinone": "reference drug",
}

exact_ranked = exact_ed50.sort_values(["ed50_ug_per_kg", "compound_id"]).copy()
exact_ranked["exact_ed50_rank"] = range(1, len(exact_ranked) + 1)
exact_ranked["key_compound_role"] = exact_ranked["compound_id"].map(key_compound_roles).fillna("")

ranking_columns = [
    "exact_ed50_rank",
    "compound_id",
    "key_compound_role",
    "ed50_ug_per_kg",
    "ed50_display",
    "iv_potency_class",
]

optional_columns = [
    "dose_mg_per_kg",
    "lv_dpdt_pct_change",
    "hr_pct_change",
    "map_pct_change",
    "curation_status",
    "curation_note",
]
ranking_columns += [col for col in optional_columns if col in exact_ranked.columns]

exact_ranked[ranking_columns]

## 7. Right-censored ED50 values

These rows should be kept visible, but they should not be treated as exact ED50 values.

For example, `>300` is a threshold statement: the true ED50 is greater than 300 μg/kg. Treating it as exactly 300 would overstate what the source table tells us.


In [ ]:
right_censored_ranked = right_censored_ed50.sort_values(["ed50_ug_per_kg", "compound_id"]).copy()
right_censored_ranked["key_compound_role"] = right_censored_ranked["compound_id"].map(key_compound_roles).fillna("")

censored_columns = [
    "compound_id",
    "key_compound_role",
    "ed50_ug_per_kg",
    "ed50_display",
    "ed50_relation",
    "ed50_censored",
    "iv_potency_class",
]

censored_columns += [col for col in optional_columns if col in right_censored_ranked.columns]

right_censored_ranked[censored_columns]

## 8. Focus compounds: 11a, Milrinone, and 23

These three compounds are useful anchors for the next phase:

- **11a**: E-1020 candidate / hydrochloride monohydrate in later context.
- **Milrinone**: reference cardiotonic agent.
- **23**: 7-yl isomer and important weak comparator for activity-cliff analysis.


In [ ]:
focus_ids = ["11a", "Milrinone", "23"]
focus = analysis_df[analysis_df["compound_id"].isin(focus_ids)].copy()
focus["key_compound_role"] = focus["compound_id"].map(key_compound_roles).fillna("")
focus = focus.sort_values("compound_id", key=lambda s: s.map({"11a": 0, "Milrinone": 1, "23": 2}).fillna(99))

focus_columns = [
    "compound_id",
    "key_compound_role",
    "ed50_display",
    "ed50_type",
    "iv_potency_class",
]
focus_columns += [col for col in ["dose_mg_per_kg", "lv_dpdt_pct_change", "hr_pct_change", "map_pct_change", "curation_status", "curation_note"] if col in focus.columns]

focus[focus_columns]

## 9. Optional: save derived ranking tables

The source curated CSV should remain the primary dataset. The following cell saves derived analysis outputs under `data/derived/` if desired.

These files are analysis outputs, not source curation files.


In [ ]:
# Optional output. Set SAVE_DERIVED_OUTPUTS = True to write files.
SAVE_DERIVED_OUTPUTS = False

if SAVE_DERIVED_OUTPUTS:
    repo_root = data_path.parent.parent.parent if data_path.parent.name == "curated" else Path(".")
    output_dir = repo_root / "data" / "derived"
    output_dir.mkdir(parents=True, exist_ok=True)

    exact_output = output_dir / "table_vi_exact_ed50_ranking_v0.csv"
    censored_output = output_dir / "table_vi_right_censored_ed50_v0.csv"

    exact_ranked.to_csv(exact_output, index=False)
    right_censored_ranked.to_csv(censored_output, index=False)

    print(f"Saved: {exact_output}")
    print(f"Saved: {censored_output}")
else:
    print("Derived outputs not saved. Set SAVE_DERIVED_OUTPUTS = True to write CSV files.")

## 10. Interpretation

This first ranking is useful, but deliberately limited.

Key points:

1. Exact ED50 values can be ranked, but right-censored values should be handled separately.
2. `>300` and `>1000` are not exact ED50 measurements.
3. `iv_potency_class` is a coarse Table VI IV potency label only.
4. ED50 ranking alone does not establish overall drug-likeness, clinical value, safety, or therapeutic superiority.
5. The next step is to compare **11a**, **Milrinone**, and **23** using more than ED50 alone, including heart-rate and mean arterial pressure effects.


## Next steps

Recommended next notebook or section:

- compare E-1020 / 11a with milrinone;
- compare 11a with compound 23;
- explain why the 11a vs 23 comparison is an activity-cliff case;
- later add Table VIII oral-duration data and Table IX PDE inhibition data.
